# E. coli protein solubility: logistic regression

Predicts whether a protein is **Soluble** (measured solubility of 50% or higher) or **Insoluble** when expressed in E. coli.

**Data files needed (put them in the same folder as this notebook):**
- `protein_solubility_master_cleaned.csv`: 4,132 proteins (3,173 labeled, 959 unlabeled)
- UniProt TSV for E. coli K-12 (proteome UP000000625) with the columns *Gene Names*, *Length* and **Sequence**

**Rules used throughout**
- Only labeled proteins are used to fit and test the model. The unlabeled proteins are only predicted.
- Train and test sets are split by gene, so the same gene never appears on both sides.
- Experimental columns in the master file (yields, chaperone solubilities) are never used, because they would leak the answer.

## 1. Setup

In [ ]:
# pip install pandas numpy scikit-learn matplotlib biopython   (run once if a package is missing)
import re, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, roc_curve, classification_report)
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer

# ---- edit these three paths if your files are somewhere else ----
MASTER_CSV = "protein_solubility_master_cleaned.csv"
UNIPROT_TSV = "uniprotkb_proteome_UP000000625_2026_10_08__1_.tsv"
OUT_DIR = "outputs/"

os.makedirs(OUT_DIR, exist_ok=True)
SEED = 42
AA = "ACDEFGHIKLMNPQRSTVWY"

## 2. Sequence features
Computes the features from each amino acid sequence: composition (all 20 amino acids), length, GRAVY, aromaticity, instability index, aliphatic index, and the percentages of charged, hydrophobic, polar, acidic and basic residues.

In [ ]:
def seq_features(seq):
    seq = "".join(c for c in str(seq).upper() if c in AA)
    n = len(seq)
    if n < 10:
        return None
    pa = ProteinAnalysis(seq)
    pct = {a: 100 * seq.count(a) / n for a in AA}            # amino-acid composition (%)
    f = {f"{a}_pct": pct[a] for a in AA}
    f["Length"] = n
    f["GRAVY"] = pa.gravy()
    f["Aromaticity"] = pa.aromaticity()
    f["Instability_Index"] = pa.instability_index()
    f["Aliphatic_Index"] = pct["A"] + 2.9 * pct["V"] + 3.9 * (pct["I"] + pct["L"])
    f["Charged_pct"] = sum(pct[a] for a in "DEKRH")
    f["Hydrophobic_pct"] = sum(pct[a] for a in "AVILMFWC")
    f["Polar_pct"] = sum(pct[a] for a in "STNQ")
    f["Acidic_pct"] = pct["D"] + pct["E"]
    f["Basic_pct"] = pct["K"] + pct["R"] + pct["H"]
    return f

## 3. Load the data and match sequences
The UniProt file is matched to the master table by b-number (or JW ID).

In [ ]:
m = pd.read_csv(MASTER_CSV)
t = pd.read_csv(UNIPROT_TSV, sep="\t")
print("Master table:", m.shape, "| UniProt table:", t.shape)

cols = {c.lower(): c for c in t.columns}
loc_col = next((c for l, c in cols.items() if "ordered locus" in l), None) or cols["gene names"]
seq_col = cols["sequence"]

pairs = []
for i, names in enumerate(t[loc_col].fillna("")):
    for tok in re.split(r"[;,\s]+", str(names)):
        if re.fullmatch(r"b\d{4}", tok) or re.fullmatch(r"JW\d{4}", tok):
            pairs.append((tok, t[seq_col].iloc[i]))
seqmap = dict(pd.DataFrame(pairs, columns=["key", "Sequence"]).drop_duplicates("key").values)

m["Sequence"] = m["B number"].map(seqmap).fillna(m["JW_ID"].map(seqmap))
print(f"Proteins with a matching sequence: {m['Sequence'].notna().sum()} / {len(m)}")

In [ ]:
feat = m["Sequence"].apply(lambda s: seq_features(s) if isinstance(s, str) else None)
F = pd.DataFrame([f if f else {} for f in feat], index=m.index)
m = pd.concat([m, F], axis=1)
m = m[m["Length"].notna()].copy()              # drop proteins with no UniProt sequence

m["MW_kDa"] = m["Calculated MW (kDa)"]
m["pI"] = m["Calculated pI"]
m["log_MW"] = np.log(m["MW_kDa"])
m["log_Length"] = np.log(m["Length"])
m["pI_dist7"] = (m["pI"] - 7).abs()            # solubility tends to drop when pI is near neutral pH
m["Cell_location"] = m["Cell location"].fillna("Unknown")
m["Gene_type"] = m["Type of gene product"].fillna("Unknown")
for c in ["Cell_location", "Gene_type"]:      # group rare categories so one-hot columns stay stable
    vc = m[c].value_counts()
    m[c] = m[c].where(m[c].isin(vc[vc >= 30].index), "Other")
m[["B number", "MW_kDa", "pI", "Length", "GRAVY", "Cell_location", "Gene_type"]].head()

## 4. Labeled vs unlabeled, and the train/test split
Target: **Soluble = 1** when `Solubility (%)` is 50 or higher. The split is grouped by gene.

In [ ]:
lab = m[m["has_solubility_label"] == True].copy()
unl = m[m["has_solubility_label"] != True].copy()
lab["y"] = (lab["Solubility (%)"] >= 50).astype(int)
assert (lab["y"] == lab["soluble_label_50"].astype(int)).all()      # label matches the 50% rule

shared = set(lab["B number"]) & set(unl["B number"])                # keep the two sets separate
if shared:
    print("Dropping from unlabeled (gene already in labeled set):", sorted(shared))
    unl = unl[~unl["B number"].isin(shared)]

seq_num = ["log_Length", "GRAVY", "Aromaticity", "Instability_Index", "Aliphatic_Index", "Charged_pct",
           "Hydrophobic_pct", "Polar_pct", "Acidic_pct", "Basic_pct"] + [f"{a}_pct" for a in AA]
num_cols = ["log_MW", "pI"] + seq_num
num_cols_v2 = num_cols + ["pI_dist7"]
cat_cols = ["Cell_location", "Gene_type"]

X, y, g = lab[num_cols_v2 + cat_cols], lab["y"], lab["B number"]
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED).split(X, y, g))
assert not set(g.iloc[tr]) & set(g.iloc[te])                        # no gene on both sides
print(f"Train {len(tr)} | Test {len(te)} | To predict (unlabeled) {len(unl)}")
print("Share of Soluble in labeled data: %.1f%%" % (100 * y.mean()))

## 5. Model-building helpers
`make_pipe` builds the pipeline: scale the numeric features, optionally add smooth spline terms (to let MW and pI act non-linearly), one-hot encode the categories, then fit the logistic regression.

In [ ]:
def make_pipe(nc, cc, C=1.0, spl=None):
    tf = [("n", StandardScaler(), nc)]
    if spl:   # cubic B-splines for the listed columns
        tf.append(("s", Pipeline([("sp", SplineTransformer(n_knots=5, degree=3)), ("sc", StandardScaler())]), spl))
    if cc:
        tf.append(("c", OneHotEncoder(handle_unknown="ignore"), cc))
    return Pipeline([("pre", ColumnTransformer(tf)), ("clf", LogisticRegression(C=C, max_iter=5000))])


def evaluate(nc, cc, name, tune=False, spl=None):
    p = make_pipe(nc, cc, spl=spl)
    if tune:   # choose the regularisation strength C with gene-grouped 5-fold CV on the training set
        gs = GridSearchCV(p, {"clf__C": [0.003, 0.01, 0.03, 0.1, 0.3, 1, 10]},
                          cv=GroupKFold(5), scoring="roc_auc", n_jobs=-1)
        gs.fit(X.iloc[tr][nc + cc], y.iloc[tr], groups=g.iloc[tr])
        p, cvauc = gs.best_estimator_, gs.best_score_
        print(f"  best C = {gs.best_params_['clf__C']}")
    else:
        p.fit(X.iloc[tr][nc + cc], y.iloc[tr]); cvauc = np.nan
    pr = p.predict_proba(X.iloc[te][nc + cc])[:, 1]
    r = dict(model=name, accuracy=accuracy_score(y.iloc[te], pr > .5),
             precision=precision_score(y.iloc[te], pr > .5), recall=recall_score(y.iloc[te], pr > .5),
             f1=f1_score(y.iloc[te], pr > .5), roc_auc=roc_auc_score(y.iloc[te], pr), cv_auc_train=cvauc)
    return p, pr, r

## 6. Compare four versions of the model
Each version is scored on the held-out test set and with 3 x repeated 5-fold gene-grouped cross-validation on all labeled data. The cross-validation numbers are the steadier ones.

In [ ]:
SPL = ["log_MW", "pI"]
variants = {
    "Baseline: MW+pI+location+type":   dict(nc=["log_MW", "pI"], spl=None, tune=False),
    "FULL: sequence-derived features": dict(nc=num_cols,         spl=None, tune=True),
    "FULL + |pI-7|":                   dict(nc=num_cols_v2,      spl=None, tune=True),
    "FULL + |pI-7| + splines(MW,pI)":  dict(nc=num_cols_v2,      spl=SPL,  tune=True),
}
res, fitted = [], {}
for name, v in variants.items():
    print(name)
    est, pr, r = evaluate(v["nc"], cat_cols, name, tune=v["tune"], spl=v["spl"])
    C = est.named_steps["clf"].C
    aucs, accs = [], []
    for seed in range(3):
        rng = np.random.default_rng(seed)
        ug = g.unique()
        fold = dict(zip(rng.permutation(ug), np.arange(len(ug)) % 5))
        f = g.map(fold).values
        oof = np.zeros(len(X))
        for k in range(5):
            mdl = make_pipe(v["nc"], cat_cols, C=C, spl=v["spl"]).fit(X[f != k][v["nc"] + cat_cols], y[f != k])
            oof[f == k] = mdl.predict_proba(X[f == k][v["nc"] + cat_cols])[:, 1]
        aucs.append(roc_auc_score(y, oof)); accs.append(accuracy_score(y, oof > .5))
    r["repCV_auc"], r["repCV_acc"] = np.mean(aucs), np.mean(accs)
    res.append(r); fitted[name] = (est, pr, v)

comp = pd.DataFrame(res).round(4)
comp.to_csv(OUT_DIR + "model_comparison.csv", index=False)
comp.drop(columns=["precision", "recall"])

## 7. Pick the best model and evaluate it on the test set

In [ ]:
best_name = comp.sort_values("repCV_auc", ascending=False).iloc[0]["model"]
print("Selected model (best repeated-CV AUC):", best_name)
model, prob, bv = fitted[best_name]
num_sel, spl_sel = bv["nc"], bv["spl"]

pred = (prob > .5).astype(int)
print(classification_report(y.iloc[te], pred, target_names=["Insoluble", "Soluble"]))
cm = confusion_matrix(y.iloc[te], pred)
print("Confusion matrix [[TN FP], [FN TP]]:\n", cm)
print("Test accuracy: %.1f%%" % (100 * accuracy_score(y.iloc[te], pred)))

## 8. Which features matter
Coefficients are for standardised features, so they are comparable. A positive value means more likely soluble.

In [ ]:
names = model.named_steps["pre"].get_feature_names_out()
coef = pd.DataFrame({"feature": names, "coef": model.named_steps["clf"].coef_[0]})
coef["odds_ratio_per_SD"] = np.exp(coef["coef"])
coef = coef.reindex(coef["coef"].abs().sort_values(ascending=False).index)
coef.to_csv(OUT_DIR + "model_coefficients.csv", index=False)
coef.head(15)

## 9. Plots

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5.5))

fpr, tpr, _ = roc_curve(y.iloc[te], prob)
ax[0].plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc_score(y.iloc[te], prob):.3f}")
ax[0].plot([0, 1], [0, 1], "k--")
ax[0].set(title="ROC curve (test set)", xlabel="False positive rate", ylabel="True positive rate")
ax[0].legend()

ax[1].imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    ax[1].text(j, i, v, ha="center", va="center", fontsize=14)
ax[1].set(title="Confusion matrix", xticks=[0, 1], yticks=[0, 1],
          xticklabels=["Insoluble", "Soluble"], yticklabels=["Insoluble", "Soluble"],
          xlabel="Predicted", ylabel="Actual")

top = coef.head(15).iloc[::-1]
ax[2].barh(top["feature"].str.replace("n__", "").str.replace("c__", ""), top["coef"])
ax[2].axvline(0, color="k", lw=.8)
ax[2].set(title="Top 15 standardised coefficients (+ = more soluble)")

plt.tight_layout()
plt.savefig(OUT_DIR + "model_evaluation.png", dpi=150)
plt.show()

## 10. Final model and predictions for the unlabeled proteins
The chosen model is refit on **all** labeled proteins, then used to predict the unlabeled ones.

Caution: the unlabeled set contains many more membrane proteins than the training set, so its predictions are less reliable than the test-set scores suggest.

In [ ]:
final = make_pipe(num_sel, cat_cols, C=model.named_steps["clf"].C, spl=spl_sel).fit(X[num_sel + cat_cols], y)
p = final.predict_proba(unl[num_sel + cat_cols])[:, 1]

out = unl[["B number", "Gene name K-12", "MW_kDa", "pI", "Cell_location", "Gene_type"]].copy()
out["P_soluble"] = p.round(4)
out["Predicted"] = np.where(p > .5, "Soluble", "Insoluble")
out.to_csv(OUT_DIR + "predictions_unlabeled.csv", index=False)
print(out["Predicted"].value_counts())
out.head(10)